In [4]:
import requests


class SearchWikipedia:
    def search(self, search: str) -> str:
        """Search Wikipedia using the MediaWiki API."""
        response = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "format": "json",
                "list": "search",
                "srsearch": search,
            },
            headers={"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"},
            timeout=10,
        )
        response.raise_for_status()
        return response.json()

    def getpage(self, pagetitle: str) -> str:
        """Return the raw content of a Wikipedia page."""
        response = requests.get(
            "https://en.wikipedia.org/w/index.php",
            params={"title": pagetitle, "action": "raw"},
            headers={"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"},
            timeout=10,
        )
        response.raise_for_status()
        return response.text

In [ ]:
wiki = SearchWikipedia()
result = wiki.search("capybara")
len(result["query"]["search"])

[{'ns': 0,
  'title': 'Capybara',
  'pageid': 6776,
  'size': 44449,
  'wordcount': 4312,
  'snippet': 'The <span class="searchmatch">capybara</span> (/kæpɪˈbɑːrə/ kap-ih-BAR-uh) or greater <span class="searchmatch">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South',
  'timestamp': '2026-09-25T20:39:36Z'},
 {'ns': 0,
  'title': 'Lesser capybara',
  'pageid': 23188846,
  'size': 5681,
  'wordcount': 582,
  'snippet': 'The lesser <span class="searchmatch">capybara</span> (Hydrochoerus isthmius) is a large semi-aquatic rodent found in South America that has vast similarities with, yet subtle differences',
  'timestamp': '2026-08-14T00:29:52Z'},
 {'ns': 0,
  'title': 'Capybara (disambiguation)',
  'pageid': 69085306,
  'size': 535,
  'wordcount': 101,
  'snippet': 'Look up <span class="searchmatch">capybara</span>\xa0or <span class="searchmatch">Capybara</span> in Wiktionary, the free dictionary. The <span class="searchmatch">capybar

In [11]:
wiki = SearchWikipedia()
result = wiki.search("capybara")
[page["title"] for page in result["query"]["search"] if "capybara" in page["title"].lower()]

['Capybara',
 'Lesser capybara',
 'Capybara (disambiguation)',
 'Capybara Games',
 'Capybara Party',
 'Capybara (software)']

In [12]:
result = wiki.getpage("Capybara")
len(result)

44313

In [13]:
from toyaikit.tools import get_instance_methods

tools = get_instance_methods(wiki)

In [14]:
instructions = """
You're a documentation assistant.

Answer the user question using only the wikipedia content base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant wikipedia articless.
   - Explain (in 2â€“3 sentences) why this search query is appropriate for the user search pattern.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the titles of wikipedia pages, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more getpage calls to retrieve the full content of the most relevant wikipedia articles.
   - For each search or getpage call, explain (in 2â€“3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved wikipedia article contents from getpage.
   - Synthesize the information from these articles into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or page retrieval.
- Use only facts found in the wikipedia knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved articles, clearly inform the user.

Additional notes:
- Prefer retrieving and analyzing full articles (via getpage) before producing the final answer.
"""

In [16]:
from pydantic_ai.messages import FunctionToolCallEvent

class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        # Detect nested streams
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)

In [ ]:
from pydantic_ai import Agent

search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [17]:
callback = NamedCallback(search_agent)

In [18]:
result = await search_agent.run(
    "talk me about capybaras",
    event_stream_handler=callback
)
result

                 pydantic-ai v2.55.0 • Python 3.13.15
      / \
     /   \       agent: search • model: openai:gpt-4o-mini • tools: 2 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


TOOL CALL (search): search({"search":"capybaras"})
TOOL CALL (search): getpage({"pagetitle":"Capybara"})
TOOL CALL (search): getpage({"pagetitle":"Lesser capybara"})


AgentRunResult(output='### Overview of Capybaras\n\n**Capybara (Hydrochoerus hydrochaeris)** is the largest living rodent, native to South America, except for Chile. These semi-aquatic mammals inhabit savannas and dense forests, often found near bodies of freshwater. Known for their social behavior, capybaras typically live in groups ranging from 10 to 20, but can come together in much larger groups during dry spells when water availability is limited.\n\nCapybaras are herbivores, primarily grazing on grasses and aquatic plants. Their physical features include a barrel-shaped body, blunt snout, and partially webbed feet, which aid their swimming capabilities. They have a lifespan of about 4 years in the wild, often longer in captivity.\n\n**Social Behavior and Communication**\nCapybaras utilize various sounds for communication, including barks and whistles, especially during mating rituals conducted in water. They have distinctive social structures with clear dominance hierarchies amon

In [20]:
question = "What is this page about? https://en.wikipedia.org/wiki/Capybara"

In [21]:
result = await search_agent.run(
    question,
    event_stream_handler=callback
)
result.output

TOOL CALL (search): search({"search":"Capybara"})
TOOL CALL (search): getpage({"pagetitle":"Capybara"})


'The Wikipedia page on the **capybara** describes this animal as the largest living rodent, scientifically known as *Hydrochoerus hydrochaeris*. Native to all South American countries except Chile, capybaras are semi-aquatic herbivores typically found near bodies of freshwater. They live in social groups, often consisting of 10 to 20 individuals, but can sometimes number over 100. The article further covers their ecology, diet, communication methods, and their interactions with humans and other species.\n\nCapybaras are not considered threatened, although they are hunted in some areas for their meat and skins. In pop culture, they have gained a following, particularly in Japan, where they are associated with calmness and harmoniousness.'

In [22]:
question = "What are the main threats to capybara populations?"
result = await search_agent.run(
    question,
    event_stream_handler=callback
)
result.output

TOOL CALL (search): search({"search":"capybara population threats"})
TOOL CALL (search): search({"search":"capybara conservation status"})
TOOL CALL (search): search({"search":"capybara threats and predators"})
TOOL CALL (search): getpage({"pagetitle":"Rodent"})


'The search for "capybara population threats" and "capybara conservation status" led to the discovery of relevant articles that discuss the threats to capybara populations. Among them, the article on **Capybara** itself provides comprehensive details about their population status, habitat, and key threats faced.\n\n### Main Threats to Capybara Populations\n\nThe main threats to capybara populations include:\n\n1. **Habitat Loss**: Expansion of agriculture and urban areas has led to significant habitat destruction, impacting their natural living conditions.\n\n2. **Hunting**: Capybaras are often hunted for their meat and pelts, which poses a direct threat to their populations.\n\n3. **Predation**: Natural predators, such as jaguars and caimans, can significantly affect local capybara populations, especially in certain habitats.\n\n4. **Environmental Changes**: Climate change and environmental degradation can alter their habitats, making them less suitable for survival.\n\n5. **Human-Wil